# 03 - Indicadores Gold Pix

Este notebook cria indicadores mensais do Pix prontos para consumo analitico.

## Camada Gold

A camada Gold contem dados agregados e orientados ao consumo. O ticket medio representa o valor medio por transacao. O crescimento mes contra mes compara o valor atual com o mes anterior usando Window Functions do Spark.

In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd().resolve().parent if Path.cwd().resolve().name in {"notebooks", "i_notebooks"} else Path.cwd().resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))


In [ ]:
from pyspark.sql import Window
from pyspark.sql import functions as F

from src.config import PIX_CLEAN_DIR, PIX_MONTHLY_INDICATORS_CSV_DIR, PIX_MONTHLY_INDICATORS_DIR, create_project_directories
from src.data_quality import ensure_columns, ensure_not_empty
from src.spark_session import get_spark_session

create_project_directories(verbose=False)
spark = get_spark_session("03-gold-indicators-pix")

In [ ]:
silver_df = spark.read.parquet(str(PIX_CLEAN_DIR))
ensure_not_empty(silver_df, "Silver Pix clean")
ensure_columns(silver_df, ["ano_mes", "quantidade_transacoes", "valor_total"], "Silver Pix clean")
silver_df.show(5, truncate=False)

In [ ]:
monthly_df = (
    silver_df
    .groupBy("ano_mes")
    .agg(
        F.sum("quantidade_transacoes").alias("quantidade_transacoes"),
        F.sum("valor_total").alias("valor_total"),
    )
    .withColumn(
        "ticket_medio",
        F.when(F.col("quantidade_transacoes") > 0, F.col("valor_total") / F.col("quantidade_transacoes"))
    )
)

window = Window.orderBy("ano_mes")
monthly_df = (
    monthly_df
    .withColumn("quantidade_mes_anterior", F.lag("quantidade_transacoes").over(window))
    .withColumn("valor_mes_anterior", F.lag("valor_total").over(window))
    .withColumn(
        "crescimento_qtd_mes_anterior",
        F.when(
            F.col("quantidade_mes_anterior") > 0,
            ((F.col("quantidade_transacoes") - F.col("quantidade_mes_anterior")) / F.col("quantidade_mes_anterior")) * 100,
        )
    )
    .withColumn(
        "crescimento_valor_mes_anterior",
        F.when(
            F.col("valor_mes_anterior") > 0,
            ((F.col("valor_total") - F.col("valor_mes_anterior")) / F.col("valor_mes_anterior")) * 100,
        )
    )
    .drop("quantidade_mes_anterior", "valor_mes_anterior")
    .orderBy("ano_mes")
)

ensure_not_empty(monthly_df, "Gold indicadores mensais")

In [ ]:
monthly_df.printSchema()
monthly_df.show(20, truncate=False)

In [ ]:
monthly_df.write.mode("overwrite").parquet(str(PIX_MONTHLY_INDICATORS_DIR))
monthly_df.coalesce(1).write.mode("overwrite").option("header", True).csv(str(PIX_MONTHLY_INDICATORS_CSV_DIR))
print(f"Indicadores Gold gravados em: {PIX_MONTHLY_INDICATORS_DIR.relative_to(PROJECT_DIR)}")
print(f"CSV analitico gravado em: {PIX_MONTHLY_INDICATORS_CSV_DIR.relative_to(PROJECT_DIR)}")

In [ ]:
spark.stop()